# AI Synthetic Data Generation & Validation Platform

A Python notebook for generating synthetic tabular data and validating schema, quality, statistical similarity, relationships, and basic privacy-risk signals. Uses a synthetic demo dataset by default.

> Synthetic data is not automatically anonymous. This educational baseline does not implement differential privacy or guarantee protection against re-identification.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

rng = np.random.default_rng(42)
n = 1200
age = rng.integers(18, 75, n)
tenure = np.clip(rng.gamma(2.5, 18, n), 0, 120).round(1)
spend = np.clip(rng.normal(75 + .35*tenure, 22, n), 10, 250).round(2)
region = rng.choice(["North","South","East","West"], n, p=[.25,.30,.20,.25])
plan = rng.choice(["Basic","Plus","Premium"], n, p=[.45,.35,.20])
p = 1/(1+np.exp(-(-2 + .025*(spend-80) - .012*tenure)))
churn = rng.binomial(1, p)
source = pd.DataFrame({"age":age,"tenure_months":tenure,"monthly_spend":spend,
                       "region":region,"plan":plan,"churn":churn})
for c in ["monthly_spend","region"]:
    source.loc[rng.choice(source.index, int(.025*n), replace=False), c] = np.nan
source.head()

## 1. Profile source data

In [ ]:
profile = pd.DataFrame({
    "dtype": source.dtypes.astype(str),
    "missing_count": source.isna().sum(),
    "missing_fraction": source.isna().mean(),
    "unique_values": source.nunique(dropna=True)
})
display(profile)
display(source.describe(include="all").T)

## 2. Generate synthetic data
This baseline samples numeric columns from clipped Gaussian distributions and categorical columns from smoothed empirical frequencies. It approximates missingness independently and may not preserve complex dependencies.

In [ ]:
def generate_synthetic(df, n_rows=None, seed=123):
    gen = np.random.default_rng(seed)
    n_rows = n_rows or len(df)
    result = {}
    for col in df.columns:
        s = df[col]
        obs = s.dropna()
        miss = float(s.isna().mean())
        if pd.api.types.is_numeric_dtype(s):
            if obs.empty:
                vals = np.full(n_rows, np.nan)
            else:
                sd = float(obs.std()) if len(obs) > 1 else 0.0
                vals = gen.normal(float(obs.mean()), sd, n_rows) if sd > 0 else np.full(n_rows, float(obs.mean()))
                vals = np.clip(vals, float(obs.min()), float(obs.max()))
                if pd.api.types.is_integer_dtype(s):
                    vals = np.rint(vals).astype(int)
        else:
            freq = obs.value_counts(normalize=True)
            cats = freq.index.to_list()
            probs = freq.to_numpy(dtype=float) + .01
            probs /= probs.sum()
            vals = gen.choice(cats, n_rows, p=probs) if cats else np.full(n_rows, None, dtype=object)
        if miss:
            vals = np.asarray(vals, dtype=object)
            vals[gen.random(n_rows) < miss] = np.nan
        result[col] = vals
    return pd.DataFrame(result, columns=df.columns)

synthetic = generate_synthetic(source, seed=42)
display(synthetic.head())
print("Synthetic shape:", synthetic.shape)

## 3. Schema and data-quality validation

In [ ]:
def validate_schema(real, synth):
    missing = sorted(set(real.columns)-set(synth.columns))
    extra = sorted(set(synth.columns)-set(real.columns))
    type_match = {c: str(real[c].dtype)==str(synth[c].dtype) for c in real.columns if c in synth}
    return {"same_columns": not missing and not extra, "missing_columns":missing,
            "extra_columns":extra, "dtype_match":type_match,
            "all_dtypes_match":all(type_match.values()) if type_match else False}

def validate_quality(df):
    nums = df.select_dtypes(include=np.number)
    return {"rows":len(df), "duplicate_rows":int(df.duplicated().sum()),
            "missing_fraction":df.isna().mean().to_dict(),
            "infinite_values":{c:int(np.isinf(nums[c].dropna()).sum()) for c in nums}}

schema_report = validate_schema(source, synthetic)
quality_report = validate_quality(synthetic)
print(json.dumps(schema_report, indent=2))
print(json.dumps(quality_report, indent=2, default=str))

## 4. Statistical similarity checks
Numeric columns are compared using normalized mean difference. Categorical columns use total variation distance. Lower gaps indicate closer marginal distributions, but do not establish downstream usefulness or privacy.

In [ ]:
def compare_marginals(real, synth):
    rows=[]
    for col in real.columns:
        r, s = real[col].dropna(), synth[col].dropna()
        if pd.api.types.is_numeric_dtype(real[col]):
            scale = float(r.std()) if len(r)>1 and r.std()>0 else 1.0
            gap = abs(float(r.mean())-float(s.mean()))/scale if len(r) and len(s) else np.nan
            rows.append({"column":col,"type":"numeric","similarity_gap":gap,
                         "real_mean":r.mean(),"synthetic_mean":s.mean()})
        else:
            cats=set(r.astype(str)) | set(s.astype(str))
            rp=r.astype(str).value_counts(normalize=True).reindex(cats,fill_value=0)
            sp=s.astype(str).value_counts(normalize=True).reindex(cats,fill_value=0)
            rows.append({"column":col,"type":"categorical",
                         "similarity_gap":.5*float((rp-sp).abs().sum()),
                         "real_mean":None,"synthetic_mean":None})
    return pd.DataFrame(rows)

similarity = compare_marginals(source, synthetic)
display(similarity)

## 5. Relationship preservation

In [ ]:
numeric_cols = source.select_dtypes(include=np.number).columns.tolist()
real_corr = source[numeric_cols].corr()
synth_corr = synthetic[numeric_cols].corr()
corr_gap = (real_corr-synth_corr).abs()
display(real_corr)
display(synth_corr)
upper = corr_gap.to_numpy()[np.triu_indices_from(corr_gap, k=1)]
mean_corr_gap = float(np.nanmean(upper)) if len(upper) else 0.0
print("Mean absolute correlation gap:", mean_corr_gap)
fig, axes = plt.subplots(1,2,figsize=(10,4))
axes[0].imshow(real_corr, vmin=-1, vmax=1); axes[0].set_title("Source")
axes[1].imshow(synth_corr, vmin=-1, vmax=1); axes[1].set_title("Synthetic")
for ax in axes:
    ax.set_xticks(range(len(numeric_cols)), numeric_cols, rotation=45, ha="right")
    ax.set_yticks(range(len(numeric_cols)), numeric_cols)
plt.tight_layout(); plt.show()

## 6. Basic nearest-neighbour risk signal
This diagnostic is a warning signal only. It is not a formal privacy test, anonymity guarantee, or differential-privacy mechanism.

In [ ]:
def nearest_neighbor_signal(real, synth):
    combined = pd.concat([real, synth], ignore_index=True)
    num_cols = combined.select_dtypes(include=np.number).columns.tolist()
    cat_cols = [c for c in combined.columns if c not in num_cols]
    blocks=[]
    if num_cols:
        x=combined[num_cols].copy()
        for c in num_cols: x[c]=x[c].fillna(x[c].median())
        blocks.append(StandardScaler().fit_transform(x))
    if cat_cols:
        x=combined[cat_cols].fillna("__MISSING__").astype(str)
        blocks.append(pd.get_dummies(x, dtype=float).to_numpy())
    matrix=np.concatenate(blocks,axis=1)
    real_x, synth_x = matrix[:len(real)], matrix[len(real):]
    distances,_=NearestNeighbors(n_neighbors=1).fit(real_x).kneighbors(synth_x)
    return {"mean_nearest_source_distance":float(distances.mean()),
            "median_nearest_source_distance":float(np.median(distances)),
            "minimum_nearest_source_distance":float(distances.min()),
            "note":"Heuristic only; not a formal privacy assessment."}

privacy_signal=nearest_neighbor_signal(source,synthetic)
print(json.dumps(privacy_signal,indent=2))

## 7. Consolidated report and exports

In [ ]:
report = {
    "source_rows":int(len(source)), "synthetic_rows":int(len(synthetic)),
    "columns":source.columns.tolist(), "schema":schema_report,
    "quality":quality_report,
    "distribution_similarity":similarity.replace({np.nan:None}).to_dict(orient="records"),
    "mean_absolute_correlation_gap":mean_corr_gap,
    "privacy_diagnostic":privacy_signal,
    "limitations":[
        "Synthetic data is not automatically anonymous.",
        "This baseline does not implement differential privacy.",
        "Independent column sampling may weaken feature relationships.",
        "Human, domain, and privacy review are required before downstream use."
    ]
}
out=Path("synthetic_data_outputs"); out.mkdir(exist_ok=True)
synthetic.to_csv(out/"synthetic_dataset.csv",index=False)
similarity.to_csv(out/"distribution_similarity.csv",index=False)
(out/"validation_report.json").write_text(json.dumps(report,indent=2,default=str),encoding="utf-8")
print("Exported:", [p.name for p in out.iterdir()])
print(json.dumps(report,indent=2,default=str)[:4500])

## 8. Production-readiness checklist

- Define utility requirements and intended downstream use before generation.
- Compare multiple generators and preserve important multivariate relationships.
- Validate constraints, rare categories, tails, and subgroup behavior.
- Conduct formal privacy review, including membership-inference and attribute-disclosure testing where appropriate.
- Consider differential privacy when the threat model requires a formal guarantee.
- Document provenance, generation settings, validation results, and limitations.
- Obtain domain-owner approval before using synthetic data for training, testing, or analytics.

**Limitations:** This notebook is an educational baseline, not a certified privacy solution. It does not guarantee anonymity or legal compliance.